In [ ]:
import transformers
transformers.__version__

In [ ]:
import sys
import numpy as np
import argparse
import pandas as pd
import pickle
import os
import torch
from pathlib import Path
import random
from time import time
import pickle

In [ ]:
from huggingface_hub import login
hf_key = "HUGGING_FACE_API_KEY"
login(token=hf_key)
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
dataset = 'bank_marketing'

dataset_folder = f"../datasets/{dataset}"

checkpoints_folder = f"{dataset_folder}/checkpoints"
data_folder = f"{dataset_folder}/data"
synthetic_data_folder = f"{dataset_folder}/synthetic-data"

os.makedirs(checkpoints_folder, exist_ok=True)
os.makedirs(data_folder, exist_ok=True)
os.makedirs(synthetic_data_folder, exist_ok=True)


methods = ["great", "taptap", "tabula"]

llms = ["distilgpt2", "taptap_distill", "qwen3_03B_distil"]

gans = ["ctgan", "tvae", "tabfairgan", "ctabgan", "ctabgan_plus"]

seeds = [42, 43, 44]

for method in methods:
  os.makedirs(f"{checkpoints_folder}/{method}", exist_ok=True)
  os.makedirs(f"{synthetic_data_folder}/{method}", exist_ok=True)

  for gan in gans:
    os.makedirs(f"{checkpoints_folder}/{gan}", exist_ok=True)
    os.makedirs(f"{synthetic_data_folder}/{gan}", exist_ok=True)

  for llm in llms:
    os.makedirs(f"{synthetic_data_folder}/{method}/{llm}", exist_ok=True)
    os.makedirs(f"{checkpoints_folder}/{method}/{llm}", exist_ok=True)
    os.makedirs(f"{checkpoints_folder}/{method}/{llm}/histories", exist_ok=True)
    for seed in seeds:
      os.makedirs(f"{checkpoints_folder}/{method}/{llm}/seed{seed}", exist_ok=True)




# Great

In [ ]:
from be_great import GReaT

batch_size = 32

epochs = 50

method = 'great'

llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')


    model = GReaT(llm=llm_hugging_face,
                experiment_dir=f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                batch_size=batch_size,
                epochs=epochs,
                save_steps=5000,
                save_total_limit=2)

    trainer = model.fit(train)

    history = trainer.state.log_history
    history_df = pd.DataFrame(history)
    history_df.to_csv(f"{checkpoints_folder}/{method}/{llm}/histories/{dataset_name}_{method}_{llm}_seed{seed}_history_{epochs}.csv", index=False)


## TapTap

In [ ]:
sys.path.append('/content/drive/MyDrive/tese/code/TapTap')
from taptap.taptap import Taptap

batch_size = 32
n_steps = 10480

method = 'taptap'

target_col = 'deposit'
task = 'classification'

llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    categorical_columns = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'poutcome']

    model = Taptap(llm=llm_hugging_face,
                  experiment_dir=f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                  steps=n_steps,
                  batch_size=batch_size,
                  numerical_modeling='split',
                  save_total_limit=2,
                  gradient_accumulation_steps=gradient_accumulation_steps)

    trainer = model.fit(train, target_col=target_col, task=task)

    history = trainer.state.log_history
    history_df = pd.DataFrame(history)
    history_df.to_csv(f"{checkpoints_folder}/{method}/{llm}/histories/{dataset_name}_{method}_{llm}_seed{seed}_history_{epochs}.csv", index=False)


## Tabula

In [ ]:
sys.path.append('/content/drive/MyDrive/tese/code/Tabula')
from tabula import Tabula

batch_size = 32
epochs = 50

method = 'tabula'

llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    categorical_columns = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'poutcome']

    model = Tabula(llm=llm_hugging_face,
                experiment_dir = f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                batch_size=batch_size,
                epochs=epochs,
                save_steps=5000,
                save_total_limit=2,
                categorical_columns = categorical_columns)


    trainer = model.fit(train)

    history = trainer.state.log_history
    history_df = pd.DataFrame(history)
    history_df.to_csv(f"{checkpoints_folder}/{method}/{llm}/histories/{dataset_name}_{method}_{llm}_seed{seed}_history_{epochs}.csv", index=False)


## CTGAN

In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata


gan = "ctgan"

seeds = [42, 43, 44]
results = {}

epochs = 300
for seed in seeds:

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    model = CTGANSynthesizer(
        metadata,
        epochs=epochs,
        verbose=True
    )

    start_time = time()
    model.fit(train)
    training_time = time() - start_time

    results[seed] = {
        'training_time': training_time

    }

    model.save(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{epochs}epochs.pkl")


with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_histories.pkl", 'wb') as f:
    pickle.dump(results, f)


## TVAE

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train)

gan = "tvae"

seeds = [42, 43, 44]
results = {}

epochs = 300
for seed in seeds:

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    model = TVAESynthesizer(
        metadata,
        epochs=epochs,
        verbose=True
    )

    start_time = time()
    model.fit(train)
    training_time = time() - start_time

    results[seed] = {
        'training_time': training_time

    }

    model.save(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{epochs}epochs.pkl")


with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_histories.pkl", 'wb') as f:
    pickle.dump(results, f)


## TabFairGAN

In [ ]:
from tabfairgan import TFG

gan = "tabfairgan"

seeds = [42, 43, 44]
results = {}

epochs = 300

for seed in seeds:


    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    train['age_group'] = (train['age'] > 25).map({True: 'older', False: 'younger'})

    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    fairness_config = {
        'fair_epochs': 100,
        'lamda': 0.5,
        'S': 'age_group',
        'Y': 'deposit',
        'S_under': 'younger',
        'Y_desire': 'yes'
    }


    model = TFG(
        df=train,
        epochs=epochs,
        batch_size=500,
        device='cuda:0',
        fairness_config=fairness_config
    )

    start_time = time()
    model.train()
    training_time = time() - start_time

    results[seed] = {
        'training_time': training_time

    }

    with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{epochs}epochs.pkl", "wb") as f:
      pickle.dump(model, f)

with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_histories.pkl", 'wb') as f:
    pickle.dump(results, f)



## CTAB-GAN

In [ ]:
from model.ctabgan import CTABGAN

gan = "ctabgan"

seeds = [42, 43, 44]
results = {}

epochs = 300
for seed in seeds:


    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    mixed_columns = {
      'pdays': [-1]
    }

    categorical_columns = [
      'job', 'marital', 'education', 'default', 'housing', 'loan',
      'contact', 'month', 'poutcome', 'deposit'
      ]

    integer_columns = [
      'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'
    ]


    model = CTABGAN(
        raw_csv_path=f'{data_folder}/{dataset_name}_seed{seed}_train.csv',
        test_ratio=0.2,
        categorical_columns=categorical_columns,
        integer_columns=integer_columns,
        log_columns=[],
        mixed_columns=mixed_columns,
        problem_type={"Classification": "deposit"},
        epochs=epochs
    )

    start_time = time()
    model.fit()
    training_time = time() - start_time

    results[seed] = {
        'training_time': training_time
    }

    with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{epochs}epochs.pkl", "wb") as f:
      pickle.dump(model, f)

with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_histories.pkl", 'wb') as f:
    pickle.dump(results, f)

## CTAB-GAN-Plus

In [ ]:
from model.ctabgan import CTABGAN

gan = "ctabgan_plus"

seeds = [42, 43, 44]
results = {}

epochs = 300
for seed in seeds:

    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    mixed_columns = {
      'pdays': [-1]
    }

    categorical_columns = [
      'job', 'marital', 'education', 'default', 'housing', 'loan',
      'contact', 'month', 'poutcome', 'deposit'
      ]

    integer_columns = [
      'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'
    ]

    model = CTABGAN(
        raw_csv_path=f'{data_folder}/{dataset_name}_seed{seed}_train.csv',
        test_ratio=0.2,
        categorical_columns=categorical_columns,
        integer_columns=integer_columns,
        log_columns=[],
        mixed_columns=mixed_columns,
        problem_type={"Classification": "deposit"}

    )

    model.synthesizer.epochs = epochs

    start_time = time()
    model.fit()
    training_time = time() - start_time

    results[seed] = {
        'training_time': training_time
    }

    with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{epochs}epochs.pkl", "wb") as f:
      pickle.dump(model, f)

with open(f"{checkpoints_folder}/{gan}/{dataset_name}_{gan}_histories.pkl", 'wb') as f:
    pickle.dump(results, f)